In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/tastelet/data")
RAW_DIR = DATA_DIR / "raw"
MOVIELENS_DIR = RAW_DIR / "ml-latest-small"

if not MOVIELENS_DIR.exists():
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    !wget -q https://files.grouplens.org/datasets/movielens/ml-latest-small.zip -O /content/ml.zip
    !unzip -q /content/ml.zip -d "{RAW_DIR}"

print(sorted(p.name for p in MOVIELENS_DIR.iterdir()))

['README.txt', 'links.csv', 'movies.csv', 'ratings.csv', 'tags.csv']


In [5]:
import pandas as pd

ratings = pd.read_csv(MOVIELENS_DIR / "ratings.csv")
print(ratings.shape)

(100836, 4)


In [6]:
movies = pd.read_csv(MOVIELENS_DIR / "movies.csv")
links = pd.read_csv(MOVIELENS_DIR / "links.csv")
print(movies.shape, links.shape)

(9742, 3) (9742, 3)


In [7]:
from google.colab import userdata

TMDB_API_KEY = userdata.get("TMDB_API_KEY")
print(len(TMDB_API_KEY))

32


In [8]:
print(links["tmdbId"].isna().sum())

8


In [9]:
links = links.dropna(subset=["tmdbId"]).astype({"tmdbId": int})
print(links.shape)

(9734, 3)


In [10]:
rating_counts = ratings["movieId"].value_counts()
print((rating_counts < 5).sum())

6074


In [11]:
MIN_RATINGS_PER_MOVIE = 5

kept_movie_ids = rating_counts[rating_counts >= MIN_RATINGS_PER_MOVIE].index.intersection(links["movieId"])
ratings = ratings[ratings["movieId"].isin(kept_movie_ids)]
print(len(kept_movie_ids), ratings.shape)

3650 (90274, 4)


In [12]:
movies = movies[movies["movieId"].isin(kept_movie_ids)]
print(movies.shape)

(3650, 3)


In [13]:
movies["year"] = movies["title"].str.extract(r"\((\d{4})\)\s*$", expand=False)
print(movies["year"].isna().sum())

0


In [15]:
movies["year"] = movies["year"].astype(int)
print(movies.head(3))

   movieId                    title  \
0        1         Toy Story (1995)   
1        2           Jumanji (1995)   
2        3  Grumpier Old Men (1995)   

                                        genres  year  
0  Adventure|Animation|Children|Comedy|Fantasy  1995  
1                   Adventure|Children|Fantasy  1995  
2                               Comedy|Romance  1995  
